<p> <center><img src="img.jpg" style="height:200px"></center> </p>

<hr style="border-width:2px;border-color:#AAD">
<center><h1> Analyse et prédiction de la variabilité de la production solaire à partir de données ouvertes de la région PACA </h1></center>
<center><h2> Collecte des données météorologiques </h2></center>
<hr style="border-width:2px;border-color:#AAD">


Après avoir récupéré les données de production, les données sur la position du soleil et les données décrivant l'athmosphère, on cherche maintenant à récupérer des **informations météorologiques** complémentaires, telles que :

 - la *température* au sol (influe sur la performance des panneaux solaires) ;
 - la *vitesse du vent* au sol (influe sur la performance des panneaux solaires) ;
 - l'*humidité* (influe sur la performance des panneaux solaires) ;
 - la *nébulosité* (influe sur la lumière reçue par les panneaux solaires).

On peut trouver ces données dans un dataset mis à disposition par la NASA via une API : **POWER Hourly API** (https://power.larc.nasa.gov/), accessible par le site https://power.larc.nasa.gov/api/pages/#/Data%20Requests/single_point_data_request_api_temporal_hourly_point_get, avec une licence Creative Commons BY 4.0 (https://creativecommons.org/licenses/by/4.0/ vu sur https://registry.opendata.aws/nasa-power/).

Cette base de données a une résolution temporelle d'une heure : comme nos données ont une résolution de 30 minutes, nous devrons les interpoler.

*Note : une autre base de données qui aurait pu servir pour la température et le vent est la base cams-global-reanalysis-eac4, mais la résolution temporelle de celle ci est de 3h.*

# I - Récupération des données précédentes


Nous commencons par importer les librairies nécessaires pour manipuler nos données :

In [1]:
# Gestion des chemins
from pathlib import Path

# On importe les librairies dont on a besoin pour traiter les datasets
import pandas as pd
import numpy as np

Puis on récupère les données collectées aux étapes précédentes :

In [2]:
# Chemin vers le répertoire de données d'entrée
input_path = Path('../data/local_data/input/')

# Chemin vers le répertoire de résultats temporaires
temp_path = Path('../data/local_data/temp/')

# Chemin vers le répertoire de résultats finaux
output_path = Path('../data/local_data/output/')

# Chemin du dataset de production
input_datasets = temp_path / 'production_2020_2025.csv'

# Chemin du dataset des communes sélectionnées 
communes = output_path / 'best_communes_geo_energy.csv'

On charge le jeu de données de l'étape précédente :

In [3]:
# On récupère le dataset contenant les données de production, les données astronomiques et les données athmosphériques
df_meteo = pd.read_csv(input_datasets, parse_dates=['datetime_utc'])

# Pour la collecte des données on n'a besoin que des créneaux horaires
df_meteo = df_meteo[['datetime_utc']] 
display(df_meteo.head())


,datetime_utc
0,2019-12-31 23:00:00+00:00
1,2019-12-31 23:30:00+00:00
2,2020-01-01 00:00:00+00:00
3,2020-01-01 00:30:00+00:00
4,2020-01-01 01:00:00+00:00


# II - Lieux retenus

On récupère les coordonnées des cinq points significatifs que l'on a précédemment déterminé.

In [4]:
df_communes = pd.read_csv(communes)
display(df_communes.head())
print(df_communes.dtypes)

,cluster_geo,best_commune,code_insee,lat,lon,energie_totale,poids,prefix
0,2,Cruis,4065,44.0845,5.8397,20356525.0,0.22,cru
1,4,Saint-Étienne-le-Laus,5140,44.5075,6.1616,325158.0,0.06,sel
2,0,Saint-Vallier-de-Thiey,6130,43.6994,6.8516,344281.0,0.07,svt
3,1,Bras,83021,43.4723,5.9558,10603661.0,0.29,bra
4,3,Eygalières,13034,43.7638,4.9554,1510927.0,0.36,eyg


cluster_geo         int64
best_commune       object
code_insee          int64
lat               float64
lon               float64
energie_totale    float64
poids             float64
prefix             object
dtype: object


# III - Collecte des données météorologiques

Le jeu de données **POWER Hourly API** est accessible par le site https://power.larc.nasa.gov/api/pages/#/Data%20Requests/.

Pour faciliter la collecte des données pour chacun de nos points d'intérêts et à l'instar de ce que nous avions fait pour la collecte des données CAMS, on crée une fonction qui interroge l'API de la NASA en fonction des coordonnées géographiques d'un lieu (latitude et longitude) et des dates de début et de fin de la période à observer.

Un jeu de données est alors enregistré au niveau d'un chemin transmis à la fonction.

In [6]:
def retrieve_nasa (latitude, longitude, date_debut, date_fin, base_chemin) :
    # On formatte nos dates pour qu'elles aient la forme voulue par l'API (yyyymmdd)
    debut = "".join([n for n in date_debut if n != '-'])
    fin = "".join([n for n in date_fin if n != '-'])

    # On construit l'URL de la requete à l'API
    url = "https://power.larc.nasa.gov/api/temporal/hourly/point?start=" + str(debut) + '&end=' + str(fin)
    url += '&latitude=' + str(latitude)+'&longitude=' + str(longitude) + "&community=re"
    url += "&parameters=T2M,WS2M,CLOUD_AMT,RH2M"
    url += "&format=csv&units=metric&header=true&time-standard=utc"

    # Signification des paramètres :
    # T2M : Temperature at 2 Meters
    # WS2M : Wind Speed at 2 Meters
    # CLOUD_AMT : Total Cloud Cover
    # RH2M : Relative Humidity at 2 Meters
    

    # On lit directement le CSV depuis l'API
    df = pd.read_csv(url, skiprows=12)

    # Créer datetime correct en UTC (NASA POWER fournit YEAR, MO, DY, HR)
    df["datetime_utc"] = pd.to_datetime({"year":  df["YEAR"],
                                         "month": df["MO"],
                                         "day":   df["DY"],
                                         "hour":  df["HR"]},
                                        utc=True)

    # Renommer les colonnes en français
    df = df.rename(columns={'T2M': 'temperature',
                            'WS2M': 'vitesse_vent',
                            'CLOUD_AMT': 'nebulosite',
                            'RH2M' : "humidite"})

    # Définir datetime_utc comme index
    df = df.set_index("datetime_utc").sort_index()

    # Remplacer les valeurs manquantes (-999) par np.nan pour qu'on puisse les identifier
    df = df.replace(-999.00, np.nan)

    # Rééchantillonner toutes les 30 minutes et interpoler
    df = df.drop(columns=['YEAR', 'MO', 'DY', 'HR']).resample("30min").interpolate(numeric_only=True)
    df = df.reset_index()

    # On enregistre le dataset
    target = base_chemin.parent / (base_chemin.name + "_" + date_debut + "_" + date_fin + ".csv")
    df.to_csv(target, index=False)
    

On détermine, à partir du jeu de données précédemment collecté, la date de début et la date de fin de la période couverte.

In [8]:
# On détermine la plage de dates à requêter
# En heures UTC le dataset de départ commence le 31 décembre 2019 et se termine le 31 décembre 2024

date_debut = df_meteo.loc[0,'datetime_utc'].strftime('%Y-%m-%d')
date_fin = df_meteo.loc[df_meteo.shape[0]-1,'datetime_utc'].strftime('%Y-%m-%d')

print("Date de début :", date_debut)
print("Date de fin :", date_fin)

Date de début : 2019-12-31
Date de fin : 2026-01-31


On interroge l'API pour collecter les données athmosphérique pour chaque point d'intérêt :

In [9]:
# On interroge l'API pour récupérer les données NASA de chaque ville
# Pour chaque ville
for ville in df_communes['prefix'] :
    # On affiche où on en est rendu dans les villes
    print(ville + "...")

    # Le chemin de base du dataset temporaire sera :
    base_chemin = temp_path / ("nasa_" + ville) # La fonction de récupération ajoutera les dates et l'extension

    latitude = df_communes.loc[df_communes['prefix']==ville, 'lat'].iloc[0]
    longitude = df_communes.loc[df_communes['prefix']==ville, 'lon'].iloc[0]
    
    # Envoi de la requete
    retrieve_nasa(
        latitude, longitude,
        date_debut, date_fin,
        base_chemin)

    # On indique que la requete est terminée
    print("Ok pour", ville, '\n')
    

cru...
Ok pour cru 

sel...
Ok pour sel 

svt...
Ok pour svt 

bra...
Ok pour bra 

eyg...
Ok pour eyg 



# IV - Aggrégation des nouvelles données collectées

Les données collectées se présentent pour le moment sous la forme de fichiers csv enregistrés dans le Drive du projet, à raison d'un fichier par point d'intérêt.

Pour aggréger les nouvelles variables explicatives, on va :

 - Charger les fichiers sous forme de DataFrame Pandas ;
 - Renommer les variables en les faisant commencer par le sigle du point d'intérêt ;
 - Fusionner les jeux de données sur la base de la variable temporelle.

On charge sous forme de DataFrame Pandas les jeux de données qu'on vient de collecter, et pour les parcourir plus facilement on utilise un dictionnaire ayant pour clé le préfixe des villes d'intérêts :

In [10]:
# On crée un dictionnaire dont les clés seront les préfixes des villes
# et les valeurs les datasets collectés
all_nasa = {}

# Pour chaque ville
for ville in df_communes['prefix'] :
    # On affiche où on en est rendu dans les villes
    print(ville + "...")
    
    # Lire les fichiers
    df = pd.read_csv(
        temp_path / ('nasa_' + ville + '_' + date_debut + '_' + date_fin + '.csv'),
        parse_dates=['datetime_utc'])
    
    # Enregistrer dans all_cams
    all_nasa[ville] = df
    
    print(f"{ville} => OK\n")


cru...
cru => OK

sel...
sel => OK

svt...
svt => OK

bra...
bra => OK

eyg...
eyg => OK



On affiche l'ensemble des datasets chargés :

In [11]:
for df in all_nasa.values():
    display(df.head())

,datetime_utc,temperature,vitesse_vent,nebulosite,humidite
0,2019-12-31 00:00:00+00:00,2.370,1.030,0.730,81.03
1,2019-12-31 00:30:00+00:00,2.470,1.065,0.885,80.24
2,2019-12-31 01:00:00+00:00,2.570,1.100,1.040,79.45
3,2019-12-31 01:30:00+00:00,2.575,1.110,1.980,79.51
4,2019-12-31 02:00:00+00:00,2.580,1.120,2.920,79.57


,datetime_utc,temperature,vitesse_vent,nebulosite,humidite
0,2019-12-31 00:00:00+00:00,-0.400,0.980,83.92,53.66
1,2019-12-31 00:30:00+00:00,-0.450,1.015,42.72,52.82
2,2019-12-31 01:00:00+00:00,-0.500,1.050,1.52,51.98
3,2019-12-31 01:30:00+00:00,-0.525,1.060,36.15,51.13
4,2019-12-31 02:00:00+00:00,-0.550,1.070,70.78,50.28


,datetime_utc,temperature,vitesse_vent,nebulosite,humidite
0,2019-12-31 00:00:00+00:00,7.630,1.69,11.510,71.330
1,2019-12-31 00:30:00+00:00,7.955,1.67,9.615,69.335
2,2019-12-31 01:00:00+00:00,8.280,1.65,7.720,67.340
3,2019-12-31 01:30:00+00:00,8.400,1.60,9.250,65.995
4,2019-12-31 02:00:00+00:00,8.520,1.55,10.780,64.650


,datetime_utc,temperature,vitesse_vent,nebulosite,humidite
0,2019-12-31 00:00:00+00:00,3.64,1.530,3.57,83.67
1,2019-12-31 00:30:00+00:00,3.92,1.505,3.04,80.61
2,2019-12-31 01:00:00+00:00,4.20,1.480,2.51,77.55
3,2019-12-31 01:30:00+00:00,4.44,1.420,3.04,74.78
4,2019-12-31 02:00:00+00:00,4.68,1.360,3.57,72.01


,datetime_utc,temperature,vitesse_vent,nebulosite,humidite
0,2019-12-31 00:00:00+00:00,4.950,0.990,26.290,96.120
1,2019-12-31 00:30:00+00:00,5.060,1.000,24.855,94.295
2,2019-12-31 01:00:00+00:00,5.170,1.010,23.420,92.470
3,2019-12-31 01:30:00+00:00,5.255,0.985,28.050,91.015
4,2019-12-31 02:00:00+00:00,5.340,0.960,32.680,89.560


On renomme les variables des datasets (à l'exeption de la variable `datetime_utc` qui servira à la fusion) en les faisants débuter par le sigle du point d'intérêt correspondant :

In [12]:
# On renomme les colonnes des différents datasets pour qu'elles comprennent le nom de la ville
for ville, df in all_nasa.items() :
    mon_dico = {'temperature' : ville + '_temperature',
                'vitesse_vent' : ville + '_vitesse_vent',
                'nebulosite' : ville +'_nebulosite',
                'humidite' : ville + '_humidite'}
    df.rename(mon_dico, axis=1, inplace=True)

for df in all_nasa.values():
    display(df.head())

,datetime_utc,cru_temperature,cru_vitesse_vent,cru_nebulosite,cru_humidite
0,2019-12-31 00:00:00+00:00,2.370,1.030,0.730,81.03
1,2019-12-31 00:30:00+00:00,2.470,1.065,0.885,80.24
2,2019-12-31 01:00:00+00:00,2.570,1.100,1.040,79.45
3,2019-12-31 01:30:00+00:00,2.575,1.110,1.980,79.51
4,2019-12-31 02:00:00+00:00,2.580,1.120,2.920,79.57


,datetime_utc,sel_temperature,sel_vitesse_vent,sel_nebulosite,sel_humidite
0,2019-12-31 00:00:00+00:00,-0.400,0.980,83.92,53.66
1,2019-12-31 00:30:00+00:00,-0.450,1.015,42.72,52.82
2,2019-12-31 01:00:00+00:00,-0.500,1.050,1.52,51.98
3,2019-12-31 01:30:00+00:00,-0.525,1.060,36.15,51.13
4,2019-12-31 02:00:00+00:00,-0.550,1.070,70.78,50.28


,datetime_utc,svt_temperature,svt_vitesse_vent,svt_nebulosite,svt_humidite
0,2019-12-31 00:00:00+00:00,7.630,1.69,11.510,71.330
1,2019-12-31 00:30:00+00:00,7.955,1.67,9.615,69.335
2,2019-12-31 01:00:00+00:00,8.280,1.65,7.720,67.340
3,2019-12-31 01:30:00+00:00,8.400,1.60,9.250,65.995
4,2019-12-31 02:00:00+00:00,8.520,1.55,10.780,64.650


,datetime_utc,bra_temperature,bra_vitesse_vent,bra_nebulosite,bra_humidite
0,2019-12-31 00:00:00+00:00,3.64,1.530,3.57,83.67
1,2019-12-31 00:30:00+00:00,3.92,1.505,3.04,80.61
2,2019-12-31 01:00:00+00:00,4.20,1.480,2.51,77.55
3,2019-12-31 01:30:00+00:00,4.44,1.420,3.04,74.78
4,2019-12-31 02:00:00+00:00,4.68,1.360,3.57,72.01


,datetime_utc,eyg_temperature,eyg_vitesse_vent,eyg_nebulosite,eyg_humidite
0,2019-12-31 00:00:00+00:00,4.950,0.990,26.290,96.120
1,2019-12-31 00:30:00+00:00,5.060,1.000,24.855,94.295
2,2019-12-31 01:00:00+00:00,5.170,1.010,23.420,92.470
3,2019-12-31 01:30:00+00:00,5.255,0.985,28.050,91.015
4,2019-12-31 02:00:00+00:00,5.340,0.960,32.680,89.560


On fusionne les datasets en se basant sur la variable `datetime_utc` qui contient la date et l'heure des observations :  

In [14]:
# On va merger sur les colonnes 'datetime_utc' en n'ajoutant aucune ligne aux données cams
for df in all_nasa.values():
    df_meteo = df_meteo.merge(right=df, on='datetime_utc', how='left')

On affiche le début du jeu de données pour avoir une première visualisation des nouvelles données :

In [15]:
# On affiche les premières lignes du dataset consolidé
display(df_meteo.head())
print("Taille du dataset :", df_meteo.shape)

,datetime_utc,cru_temperature,cru_vitesse_vent,cru_nebulosite,cru_humidite,sel_temperature,sel_vitesse_vent,sel_nebulosite,sel_humidite,svt_temperature,...,svt_nebulosite,svt_humidite,bra_temperature,bra_vitesse_vent,bra_nebulosite,bra_humidite,eyg_temperature,eyg_vitesse_vent,eyg_nebulosite,eyg_humidite
0,2019-12-31 23:00:00+00:00,2.750,1.190,0.730,76.96,0.980,1.450,22.680,52.23,9.30,...,29.440,57.600,3.230,1.620,5.700,86.880,2.720,1.220,51.750,100.000
1,2019-12-31 23:30:00+00:00,3.185,1.085,1.825,73.01,0.965,1.465,25.975,51.75,9.48,...,24.785,56.165,3.345,1.565,4.635,84.455,2.760,1.405,54.215,98.790
2,2020-01-01 00:00:00+00:00,3.620,0.980,2.920,69.06,0.950,1.480,29.270,51.27,9.66,...,20.130,54.730,3.460,1.510,3.570,82.030,2.800,1.590,56.680,97.580
3,2020-01-01 00:30:00+00:00,3.880,0.830,2.645,66.34,0.875,1.495,17.480,51.23,9.72,...,10.340,54.250,3.650,1.435,3.040,79.380,2.675,1.635,33.325,96.065
4,2020-01-01 01:00:00+00:00,4.140,0.680,2.370,63.62,0.800,1.510,5.690,51.19,9.78,...,0.550,53.770,3.840,1.360,2.510,76.730,2.550,1.680,9.970,94.550


Taille du dataset : (106704, 21)


# V - Enregistrement des données NASA collectées

Nous avons terminé la collecte des données explicatives relatives aux `conditions météorologiques` pour chaque point significatif du point de vue de la production d'énergie solaire, via le jeu de données **POWER Hourly API** de la `NASA`.

Nous enregistrons notre jeu de données actuel pour clore la collecte de données météorologiques.


In [16]:
# On enregistre ce dataset contenant l'ensemble des données souhaitées (production + astronomie + météo)
df_meteo.to_csv(temp_path / "meteorologie_2020_2025.csv", index=False)

In [19]:
# Exemple de la manière dont charger ce dataset final de données météorologiques :
df = pd.read_csv(temp_path / "meteorologie_2020_2025.csv", parse_dates=['datetime_utc'])
display(df.head())
print("Taille du dataset :", df.shape)

,datetime_utc,cru_temperature,cru_vitesse_vent,cru_nebulosite,cru_humidite,sel_temperature,sel_vitesse_vent,sel_nebulosite,sel_humidite,svt_temperature,...,svt_nebulosite,svt_humidite,bra_temperature,bra_vitesse_vent,bra_nebulosite,bra_humidite,eyg_temperature,eyg_vitesse_vent,eyg_nebulosite,eyg_humidite
0,2019-12-31 23:00:00+00:00,2.750,1.190,0.730,76.96,0.980,1.450,22.680,52.23,9.30,...,29.440,57.600,3.230,1.620,5.700,86.880,2.720,1.220,51.750,100.000
1,2019-12-31 23:30:00+00:00,3.185,1.085,1.825,73.01,0.965,1.465,25.975,51.75,9.48,...,24.785,56.165,3.345,1.565,4.635,84.455,2.760,1.405,54.215,98.790
2,2020-01-01 00:00:00+00:00,3.620,0.980,2.920,69.06,0.950,1.480,29.270,51.27,9.66,...,20.130,54.730,3.460,1.510,3.570,82.030,2.800,1.590,56.680,97.580
3,2020-01-01 00:30:00+00:00,3.880,0.830,2.645,66.34,0.875,1.495,17.480,51.23,9.72,...,10.340,54.250,3.650,1.435,3.040,79.380,2.675,1.635,33.325,96.065
4,2020-01-01 01:00:00+00:00,4.140,0.680,2.370,63.62,0.800,1.510,5.690,51.19,9.78,...,0.550,53.770,3.840,1.360,2.510,76.730,2.550,1.680,9.970,94.550


Taille du dataset : (106704, 21)


# VI - Concaténation de l'ensemble des données collectées

Pour clore la collecte de données brutes, il reste à fusionner l'ensemble des données collectées :
 * de **production** ;
 * d'**astronomie** ;
 * d'**athmosphère** et
 * de **météorologie**.

In [5]:
# Données de production :
df_prod = pd.read_csv(temp_path / 'production_2020_2025.csv', parse_dates=['datetime_utc'])

# Données astronomiques :
df_astro = pd.read_csv(temp_path / 'astronomie_2020_2025.csv', parse_dates=['datetime_utc'])

# Données athmosphériques :
df_athmo = pd.read_csv(temp_path / 'athmosphere_2020_2025.csv', parse_dates=['datetime_utc'])

# Données météorologiques :
df_meteo = pd.read_csv(temp_path / 'meteorologie_2020_2025.csv', parse_dates=['datetime_utc'])

# Fusion données production + astronomiques :
df_all = pd.merge(df_prod, df_astro, on=['datetime_utc'])

# Fusion données production + astronomiques + athmosphériques :
df_all = pd.merge(df_all, df_athmo, on=['datetime_utc'])

# Fusion données production + astronomiques + athmosphériques + météorologiques :
df_all = pd.merge(df_all, df_meteo, on=['datetime_utc'])

# Tri des colonnes par ordre alphabétique
df_all = df_all.sort_index(axis=1)

# Affichages pour info
print(f"Dimensions des datasets fusionnés : {df_all.shape}")
display(df_all.head())

Dimensions des datasets fusionnés : (106704, 86)


,bra_altitude,bra_azimuth,bra_bhi,bra_bni,bra_clear_sky_bhi,bra_clear_sky_bni,bra_clear_sky_dhi,bra_clear_sky_ghi,bra_dhi,bra_ghi,...,svt_ghi,svt_humidite,svt_nebulosite,svt_reliability,svt_temperature,svt_toa,svt_vitesse_vent,target,tch_solaire,tco_solaire
0,-68.046190,335.241040,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,57.600,29.440,1.0,9.30,0.0,1.190,0.0,0.0,0.0
1,-69.500739,353.945722,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,56.165,24.785,1.0,9.48,0.0,1.075,0.0,0.0,0.0
2,-69.141123,13.536657,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,54.730,20.130,1.0,9.66,0.0,0.960,0.0,0.0,0.0
3,-67.054165,31.240402,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,54.250,10.340,1.0,9.72,0.0,0.850,0.0,0.0,0.0
4,-63.656435,45.699703,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,53.770,0.550,1.0,9.78,0.0,0.740,0.0,0.0,0.0


# VII - Suppression des données incomplètes

Le satellite CERES qui nous a fournit les données concernant la nébulisité a cessé de collecter ces données le 30 décembre 2025. On supprime donc les données postérieures à cette date.

N.B. : l'utilisation du jeu de données https://www.ecmwf.int/en/forecasts/datasets/open-data serait peut être possible en remplacement des données de la NASA, mais la résolution temporelle est de 12h.

In [6]:
df_all = df_all.loc[df_all['datetime_utc'] < '2025-12-31']

display(df_all['datetime_utc'])


0        2019-12-31 23:00:00+00:00
1        2019-12-31 23:30:00+00:00
2        2020-01-01 00:00:00+00:00
3        2020-01-01 00:30:00+00:00
4        2020-01-01 01:00:00+00:00
                    ...           
105165   2025-12-30 21:30:00+00:00
105166   2025-12-30 22:00:00+00:00
105167   2025-12-30 22:30:00+00:00
105168   2025-12-30 23:00:00+00:00
105169   2025-12-30 23:30:00+00:00
Name: datetime_utc, Length: 105170, dtype: datetime64[ns, UTC]

# VIII - Enregistrement

On termine ce notebook par l'enregistrement des données collectées et fusionnées :

In [7]:
# On enregistre ce dataset contenant l'ensemble des données souhaitées (production + astronomie + athmosphère + météo)
df_all.to_csv(output_path / "raw_2020_2025.csv", index=False)
